# Multi-model RAG with ColPali：把 PDF 做成“图像检索”

这一节学习一种 Multi-modal RAG 路线：**不先把 PDF 里的图/表“转写成文本”再检索**，而是把页面当作图像，让检索模型直接在“图像空间”里找最相关页面，然后交给 vision LLM 基于检索到的页面回答问题。

- 你会做的事：PDF 下载 → ColPali 建索引 → 用自然语言检索出最相关页面（base64 图像）→ vision LLM 读图回答。

<img src="../images/multi_model_rag_with_colpali.svg" width="220" />

## 0) 导入依赖并加载环境变量

说明：本节不在 notebook 内安装依赖；请自行确保已安装 `byaldi`、`Pillow`、`langchain-openai` 等。


In [1]:
import base64
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Image as IPyImage, display
from PIL import Image

from byaldi import RAGMultiModalModel
from langchain_openai import ChatOpenAI

load_dotenv()

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

## 1) 准备示例 PDF

我们沿用原 notebook 的示例：下载论文 *Attention Is All You Need*（Transformer）。


In [ ]:
def download_file(url: str, dst: Path, *, timeout_s: float = 60.0) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 0:
        return dst

    r = requests.get(url, timeout=timeout_s, allow_redirects=True)
    r.raise_for_status()
    dst.write_bytes(r.content)
    return dst


PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = download_file(PDF_URL, DATA_DIR / "attention_is_all_you_need.pdf")
str(PDF_PATH)

'../data/attention_is_all_you_need.pdf'

## 2) 用 ColPali 把 PDF 索引成“可检索的页面图像”

这里的核心是 `RAGMultiModalModel`：

- `from_pretrained(...)` 加载 ColPali 检索模型
- `index(...)` 把 PDF 的页面编码进索引（可选把页面图像 base64 随索引一并保存，便于后续直接拿出来喂给 VLM）


In [ ]:
RAG = RAGMultiModalModel.from_pretrained("vidore/colpali-v1.2", verbose=1)

RAG.index(
    input_path=str(PDF_PATH),
    index_name="attention_is_all_you_need",
    store_collection_with_index=True,
    overwrite=True,
)

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


adapter_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/66.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## 3) 用自然语言检索出最相关页面（返回 base64 图像）

你可以把问题当成检索 query。ColPali 会返回最相关的页面结果；其中包含页面的 base64 图像（用于下游多模态生成）。


In [ ]:
query = "What is the main idea of the Transformer architecture introduced in this paper?"
results = RAG.search(query, k=1)
results[0]

## 4) 解码页面图像并在 notebook 中展示

这一步把检索结果中的 `base64` 解码成真实图像，方便你确认“检索到底拿回了哪一页”。


In [ ]:
image_b64 = results[0].base64
image_bytes = base64.b64decode(image_b64)

img_path = DATA_DIR / "colpali_retrieved_0.jpg"
img_path.write_bytes(image_bytes)

image = Image.open(img_path)
display(image)
str(img_path)

## 5) 把“检索到的页面图像 + 问题”交给 vision LLM 生成答案

我们用 LangChain 的标准多模态 content blocks，把 base64 图像直接作为输入之一传给支持视觉的模型。


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

message = {
    "role": "user",
    "content": [
        {"type": "text", "text": query},
        {"type": "image", "base64": image_b64, "mime_type": "image/jpeg"},
    ],
}

resp = llm.invoke([message])
resp.content